In [25]:
from xtquant import xtdata
import pandas as pd
import numpy as np

# ===================== 1. 基础参数配置 =====================
stock_list = [
    "600519.SH",  # 贵州茅台
    "000858.SZ",  # 五粮液
    "002594.SZ",  # 比亚迪
    "601318.SH",  # 中国平安
    "300750.SZ",  # 宁德时代
    "600036.SH",  # 招商银行
    "000001.SZ",  # 平安银行
    "601899.SH"   # 紫金矿业
]
start_time = "20240101"
end_time = "20241231"

table_list = ['Balance','Income']

# 下载行情、财务数据
xtdata.download_history_data2(
    stock_list=stock_list,
    period="1d",
    start_time=start_time,
    end_time=end_time
)
xtdata.download_financial_data(stock_list=stock_list, table_list=table_list)

# ===================== 2. 读取日线行情 =====================
data_dict = xtdata.get_market_data_ex(
    field_list=["close"],
    stock_list=stock_list,
    period="1d",
    start_time=start_time,
    end_time=end_time,
    count=-1,
    dividend_type="front"
)

all_price_data = []
for code, df_raw in data_dict.items():
    df = df_raw.reset_index()
    df.columns = ["date_str", "price"]
    df["symbol"] = code
    df["date"] = pd.to_datetime(df["date_str"], format="%Y%m%d")
    all_price_data.append(df[["date", "symbol", "price"]])

price_df = pd.concat(all_price_data, ignore_index=True)
print("✅ 行情数据读取完成")

# ===================== 3. 财务数据获取 + 官方接口获取总股本 =====================
def get_stock_finance(stock_code):
    # 新版标准接口获取股票基础信息，提取总股本 TotalVolume
    instrument_info = xtdata.get_instrument_detail(stock_code)
    total_share = instrument_info["TotalVolume"]

    # 获取资产负债表、利润表
    fin_res = xtdata.get_financial_data(
        stock_list=[stock_code],
        table_list=table_list,
        start_time=start_time,
        end_time=end_time
    )

    # 利润表（已验证可用字段）
    df_income = fin_res[stock_code]["Income"].reset_index()
    df_income.rename(columns={"m_timetag":"end_date"}, inplace=True)
    df_income["end_date"] = pd.to_datetime(df_income["end_date"], format="%Y%m%d")
    df_income = df_income[["end_date", "operating_revenue", "net_profit_excl_min_int_inc", "s_fa_eps_basic"]]
    df_income.columns = ["end_date", "sales", "profit", "eps"]

    # 资产负债表（已验证可用字段）
    df_balance = fin_res[stock_code]["Balance"].reset_index()
    df_balance.rename(columns={"m_timetag":"end_date"}, inplace=True)
    df_balance["end_date"] = pd.to_datetime(df_balance["end_date"], format="%Y%m%d")
    df_balance = df_balance[["end_date", "cash_equivalents", "tot_liab", "tot_shrhldr_eqy_excl_min_int"]]
    df_balance.columns = ["end_date", "cash", "debt", "total_equity"]

    # 合并财务报表
    fin_merge = pd.merge(df_income, df_balance, on="end_date", how="outer")
    # 填充总股本（静态最新值）
    fin_merge["total_share"] = total_share
    # 每股净资产 = 归属母公司权益 / 总股本
    fin_merge["book_value"] = fin_merge["total_equity"] / fin_merge["total_share"]

    return fin_merge[["end_date", "eps", "book_value", "sales", "total_share", "profit", "cash", "debt"]]

# ===================== 4. 行情财务时间对齐拼接 =====================
total_stock_list = []
for code in stock_list:
    single_price = price_df[price_df["symbol"] == code].copy().sort_values("date")
    try:
        single_fin = get_stock_finance(code).sort_values("end_date")
    except Exception as e:
        print(f"⚠️ {code} 财务数据获取失败: {e}")
        continue

    merge_df = pd.merge_asof(
        single_price,
        single_fin,
        left_on="date",
        right_on="end_date",
        direction="backward"
    )

    # 计算因子所需衍生字段
    merge_df["market_cap"] = merge_df["price"] * merge_df["total_share"]
    merge_df["dividend"] = merge_df["profit"] * 0.3 / merge_df["total_share"]
    merge_df["ebitda"] = merge_df["profit"] * 1.2

    keep_cols = [
        "date", "symbol", "price", "eps", "book_value", "sales",
        "dividend", "market_cap", "debt", "cash", "ebitda"
    ]
    merge_df = merge_df[keep_cols].dropna()
    if not merge_df.empty:
        total_stock_list.append(merge_df)

# 导出数据集
if total_stock_list:
    final_df = pd.concat(total_stock_list, ignore_index=True)
    final_df.to_csv("qmt_multi_stock.csv", index=False, encoding="utf-8-sig")
    print("✅ 多股票行情+财务数据保存成功：qmt_multi_stock.csv")
    print(final_df.head())
else:
    print("❌ 无有效财务数据，请检查时间区间或QMT权限")

✅ 行情数据读取完成
✅ 多股票行情+财务数据保存成功：qmt_multi_stock.csv
        date     symbol       price    eps  book_value  sales  dividend  \
0 2024-04-01  600519.SH  1586.91777  19.16   191.77291    0.0  5.775286   
1 2024-04-02  600519.SH  1579.57777  19.16   191.77291    0.0  5.775286   
2 2024-04-03  600519.SH  1580.69777  19.16   191.77291    0.0  5.775286   
3 2024-04-08  600519.SH  1532.24777  19.16   191.77291    0.0  5.775286   
4 2024-04-09  600519.SH  1527.80777  19.16   191.77291    0.0  5.775286   

     market_cap          debt          cash        ebitda  
0  1.983777e+12  3.698777e+10  7.419713e+10  2.887831e+10  
1  1.974601e+12  3.698777e+10  7.419713e+10  2.887831e+10  
2  1.976001e+12  3.698777e+10  7.419713e+10  2.887831e+10  
3  1.915435e+12  3.698777e+10  7.419713e+10  2.887831e+10  
4  1.909884e+12  3.698777e+10  7.419713e+10  2.887831e+10  


In [1]:
import pandas as pd

# 模拟某只股票连续5个交易日收盘价
df = pd.DataFrame({
    "symbol": ["600519.SH"] * 5,
    "date": ["2024-04-01", "2024-04-02", "2024-04-03", "2024-04-04", "2024-04-05"],
    "price": [100, 105, 107, 103, 108]
})

# 1. 先算：当日相对前一日的涨跌幅
df["daily_ret"] = df.groupby("symbol")["price"].pct_change()
print(df)
# 2. 向上偏移1行，得到【未来一天收益率】作为模型标签
df["future_ret"] = df["daily_ret"].shift(-1)

print(df)

      symbol        date  price  daily_ret
0  600519.SH  2024-04-01    100        NaN
1  600519.SH  2024-04-02    105   0.050000
2  600519.SH  2024-04-03    107   0.019048
3  600519.SH  2024-04-04    103  -0.037383
4  600519.SH  2024-04-05    108   0.048544
      symbol        date  price  daily_ret  future_ret
0  600519.SH  2024-04-01    100        NaN    0.050000
1  600519.SH  2024-04-02    105   0.050000    0.019048
2  600519.SH  2024-04-03    107   0.019048   -0.037383
3  600519.SH  2024-04-04    103  -0.037383    0.048544
4  600519.SH  2024-04-05    108   0.048544         NaN


In [3]:
#coding=utf-8
from xtquant.xttrader import XtQuantTrader
from xtquant.xttype import StockAccount

# ========= 配置项 =========
SESSION_ID = 123456   # 自定义任意数字session，随便填不重复即可
PORT = 58612          # MiniQMT右下角显示的本地端口，默认58612
ACCOUNT_ID = "testS"  # MiniQMT登录的资金账号字符串
# ==========================

# 正确构造：session_id, port 两个参数
xt_trader = XtQuantTrader(SESSION_ID, PORT)

# 启动连接
ret = xt_trader.start()
print("连接启动返回码：", ret)
if ret != 0:
    print("连接失败！检查：MiniQMT是否打开登录、端口是否正确")
else:
    print("✅ MiniQMT客户端连接成功")

# 注册消息回调（必须）
xt_trader.subscribe()

# 账户对象
stock_acc = StockAccount(ACCOUNT_ID)

# 查询资金资产
asset_list = xt_trader.query_stock_asset(stock_acc)
if not asset_list:
    print("未查询到账户资产，请核对账号ID")
else:
    asset = asset_list[0]
    print("\n==== 账户资金信息 ====")
    print(f"总资金：{asset.m_dTotal:.2f}")
    print(f"可用资金：{asset.m_dAvailable:.2f}")
    print(f"持仓市值：{asset.m_dMarketValue:.2f}")

# 查询持仓
pos_list = xt_trader.query_stock_positions(stock_acc)
print("\n==== 当前持仓 ====")
if pos_list is None or len(pos_list) == 0:
    print("暂无持仓")
else:
    for pos in pos_list:
        code = f"{pos.m_strInstrumentID}.{pos.m_strExchangeID}"
        print(f"{code} 持仓：{pos.m_nVolume} 股，现价：{pos.m_dLastPrice}")

# 断开连接
xt_trader.stop()

AttributeError: 'int' object has no attribute 'encode'